# 🗺️ View — Receita por Estado

Validação da view `vw_receita_por_estado` antes de mover para o Streamlit.

In [2]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

#formata todos os números float com 2 casas decimais na exibição do Jupyter.
pd.set_option('display.float_format', '{:.2f}'.format)

pedidos    = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
clientes   = pd.read_csv("../dados/clientes_limpo.csv")
pagamentos = pd.read_csv("../dados/pagamentos_limpo.csv")

print("Dados carregados!")

Dados carregados!


## 🧪 Testando o código antes de criar a view

In [4]:
pedidos.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13


In [5]:
clientes.head(2)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP


In [6]:
pagamentos.head(2)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39


In [9]:
## Dois colchetes — acessa MÚLTIPLAS colunas, retorna DataFrame

df = (pedidos
      .merge(clientes[['customer_id','customer_state', 'customer_city']], on= 'customer_id', how='left')
      .merge(pagamentos,on='order_id', how='left')
      )

#df = df[df['order_status'] == 'delivered'] — filtra o DataFrame: Retorna todas as linhas onde order_status é 'delivered' - Resultado: DataFrame filtrado
#df = df['order_status'] == 'delivered' — compara coluna com valor: Retorna True ou False para cada linha - Resultado: Series de booleanos

#Filtrando só pedidos entregues
df= df[df['order_status']== 'delivered']

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,payment_sequential,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,1.00,credit_card,1.00,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,3.00,voucher,1.00,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,2.00,voucher,1.00,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,1.00,boleto,1.00,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO,vianopolis,1.00,credit_card,3.00,179.12


In [15]:
#Extradindo ano e mês

df['ano'] = df['order_purchase_timestamp'].dt.year
df['data_mes'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Calculando prazo de entrega em dias antes de agrupar
df['dias_separacao']  = (df['order_delivered_carrier_date'] - 
                          df['order_purchase_timestamp']).dt.days

df['dias_transporte'] = (df['order_delivered_customer_date'] - 
                          df['order_delivered_carrier_date']).dt.days

df['dias_total']      = (df['order_delivered_customer_date'] - 
                          df['order_purchase_timestamp']).dt.days


#Agrupando por estado
receita_estado = (df.groupby(['customer_state','ano','data_mes'])
                  .agg(
                      total_pedidos = ('order_id', 'nunique'),
                      total_clientes = ('customer_id', 'nunique'),
                      receita_total = ('payment_value', 'sum'),
                      soma_dias_separacao     = ('dias_separacao', 'sum'),
                      soma_dias_transporte    = ('dias_transporte', 'sum'),
                      soma_dias_total         = ('dias_total', 'sum'),
                      total_pedidos_entregues = ('dias_total', 'nunique')
                  ).reset_index())
receita_estado['receita_total']          = receita_estado['receita_total'].round(2)
receita_estado['ticket_medio']           = (receita_estado['receita_total'] / receita_estado['total_pedidos']).round(2)
receita_estado['prazo_separacao_dias']   = (receita_estado['soma_dias_separacao'] / receita_estado['total_pedidos_entregues']).round(1)
receita_estado['prazo_transporte_dias']  = (receita_estado['soma_dias_transporte'] / receita_estado['total_pedidos_entregues']).round(1)
receita_estado['prazo_total_dias']       = (receita_estado['soma_dias_total'] / receita_estado['total_pedidos_entregues']).round(1)

receita_estado.head()

,customer_state,ano,data_mes,total_pedidos,total_clientes,receita_total,soma_dias_separacao,soma_dias_transporte,soma_dias_total,total_pedidos_entregues,ticket_medio,prazo_separacao_dias,prazo_transporte_dias,prazo_total_dias
0,AC,2017,2017-01,2,2,723.15,6.00,33.00,40.00,2,361.58,3.00,16.50,20.00
1,AC,2017,2017-02,3,3,597.40,12.00,63.00,76.00,3,199.13,4.00,21.00,25.30
2,AC,2017,2017-03,2,2,530.18,5.00,33.00,40.00,2,265.09,2.50,16.50,20.00
3,AC,2017,2017-04,5,5,1351.51,13.00,112.00,127.00,4,270.30,3.20,28.00,31.80
4,AC,2017,2017-05,8,8,2382.64,24.00,138.00,167.00,6,297.83,4.00,23.00,27.80


In [16]:
from views.vw_receita_por_estado import get_receita_por_estado

df_estado = get_receita_por_estado(pedidos, clientes, pagamentos)
df_estado.head()

,customer_state,ano,data_mes,total_pedidos,total_clientes,receita_total,soma_dias_separacao,soma_dias_transporte,soma_dias_total,total_pedidos_entregues,ticket_medio,prazo_separacao_dias,prazo_transporte_dias,prazo_total_dias
0,AC,2017,2017-01,2,2,723.15,6.00,33.00,40.00,2,361.58,3.00,16.50,20.00
1,AC,2017,2017-02,3,3,597.40,12.00,63.00,76.00,3,199.13,4.00,21.00,25.30
2,AC,2017,2017-03,2,2,530.18,5.00,33.00,40.00,2,265.09,2.50,16.50,20.00
3,AC,2017,2017-04,5,5,1351.51,13.00,112.00,127.00,5,270.30,2.60,22.40,25.40
4,AC,2017,2017-05,8,8,2382.64,24.00,138.00,167.00,8,297.83,3.00,17.20,20.90
